<a href="https://colab.research.google.com/github/bradkim1/ExprerimentWithVariousModels/blob/main/Copy_of_ExprerimentWIthVariousModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
from google.colab import drive

# Mount and Load the data file from Google Drive

drive.mount('/content/drive')

import pandas as pd

file_path = '/content/drive/MyDrive/ML_Datasets/creditcardcsv.data'
df = pd.read_csv(file_path)

# Confirm 'Class' exists
print("Columns:", df.columns.tolist())
assert 'Class' in df.columns, "'Class' column is missing."

# Split features and target
from sklearn.model_selection import train_test_split

# 'df' is preprocessed DataFrame
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train shape:", X_train.shape)
print("y_train class balance:\n", y_train.value_counts())


Mounted at /content/drive
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
X_train shape: (227845, 30)
y_train class balance:
 Class
0    227451
1       394
Name: count, dtype: int64


In [ ]:
#Baseline models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced'),
    'Random Forest': RandomForestClassifier(class_weight='balanced')
}

In [ ]:
#Advanced models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models.update({
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier()
})

In [ ]:
#Model evaluation function
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, y_pred))
    roc_score = roc_auc_score(y_test, y_proba)
    print(f'ROC AUC Score: {roc_score:.4f}')
    return roc_score

In [ ]:
#Model comparison

best_model_name = None
best_model = None
best_auc = 0

for name, model in models.items():
    print(f'--- {name} ---')
    auc = evaluate_model(model, X_train, y_train, X_test, y_test)
    print('\n')

    if auc > best_auc:
        best_auc = auc
        best_model_name = name
        best_model = model


--- Logistic Regression ---


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       1.00      0.97      0.98     56864
           1       0.05      0.92      0.10        98

    accuracy                           0.97     56962
   macro avg       0.53      0.94      0.54     56962
weighted avg       1.00      0.97      0.98     56962

ROC AUC Score: 0.9727


--- Decision Tree ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.70      0.73      0.72        98

    accuracy                           1.00     56962
   macro avg       0.85      0.87      0.86     56962
weighted avg       1.00      1.00      1.00     56962

ROC AUC Score: 0.8671


--- Random Forest ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.95      0.73      0.83        98

    accuracy                           1.00     56962
   macro avg       0.97      0.87   

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [22:34:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.92      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962

ROC AUC Score: 0.9743


--- LightGBM ---
[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.076292 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001729 -> initscore=-6.358339
[LightGBM] [Info] Start training from score -6.358339
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.37      

In [ ]:
#Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8]
}

grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced'),
    param_grid,
    scoring='roc_auc',
    cv=3
)

grid_search.fit(X_train, y_train)
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best ROC AUC: {grid_search.best_score_:.4f}')


Best parameters: {'max_depth': 6, 'n_estimators': 200}
Best ROC AUC: 0.9842


In [ ]:
#Handling class imbalance
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


In [ ]:
#Save the best-performing model for future use:
import joblib

if best_model:
    joblib.dump(best_model, f'{best_model_name.replace(" ", "_").lower()}_best_model.pkl')
    print(f" Best model '{best_model_name}' saved with ROC AUC: {best_auc:.4f}")
else:
    print(" No model was successfully trained.")


 Best model 'XGBoost' saved with ROC AUC: 0.9743
